# Congressional Voting Record Classifier Benchmark
### Decision Tree · K-Nearest Neighbours · Naive Bayes — Built From Scratch

---

**Course:** Data Warehousing and Mining — MPSTME, NMIMS  
**Academic Year:** 2025–2026  
**Author:** Sana Jadhav

---

## Overview

This notebook implements and benchmarks three supervised machine learning classifiers — Decision Tree, K-Nearest Neighbours, and Naive Bayes — built entirely from scratch in Python without any ML libraries.

**Dataset:** UCI Congressional Voting Records — 435 U.S. House of Representatives members, each described by 16 key votes, with a binary target: Democrat vs Republican.

**Objective:** Predict a legislator's party affiliation from their voting record, and rigorously compare classifier performance using a balanced analytical framework.

**What makes this benchmark meaningful:**
- All features are categorical (yes / no / abstain) — preprocessing decisions differ meaningfully per classifier
- Decision Tree feature importance reveals *which votes most strongly predict party*
- KNN uses Hamming distance (correct for binary categorical data) with k-sensitivity analysis
- Naive Bayes smoothing sensitivity (effect of Laplace α on accuracy) is analysed
- K-Fold Cross-Validation and Learning Curves applied to all three classifiers

**Pipeline:**
1. Imports
2. Data Loading & Exploration
3. Exploratory Data Analysis
4. Preprocessing & Train-Test Split
5. Decision Tree — Implementation, Training & Feature Importance
6. K-Nearest Neighbours — Implementation, Training & k-Sensitivity Analysis
7. Naive Bayes — Implementation, Training & Smoothing Sensitivity
8. Evaluation & Results
9. K-Fold Cross-Validation
10. Learning Curves
11. Final Comparison & Recommendations

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. Data Loading & Exploration

Dataset: `house-votes-84.data` (UCI Congressional Voting Records)  
435 U.S. House members with 16 binary vote features and 1 target (`party`: democrat / republican).  
Missing values are encoded as `?` (abstain) in the original file — handled as a third category.

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload house-votes-84.data

COLUMNS = [
    'party',
    'handicapped_infants', 'water_project_cost_sharing',
    'adoption_of_the_budget_resolution', 'physician_fee_freeze',
    'el_salvador_aid', 'religious_groups_in_schools',
    'anti_satellite_test_ban', 'aid_to_nicaraguan_contras',
    'mx_missile', 'immigration', 'synfuels_corporation_cutback',
    'education_spending', 'superfund_right_to_sue', 'crime',
    'duty_free_exports', 'export_administration_act_south_africa'
]

df = pd.read_csv('house-votes-84.data', header=None, names=COLUMNS)
print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Class distribution
dist = df['party'].value_counts()
print('Class Distribution:')
for label, count in dist.items():
    print(f'  {label.capitalize()}: {count} ({count/len(df)*100:.1f}%)')

print('\nUnique vote values:', df.iloc[:, 1:].stack().unique())
print('\nMissing (?) counts per feature:')
for col in COLUMNS[1:]:
    n = (df[col] == '?').sum()
    if n > 0:
        print(f'  {col}: {n}')

## 3. Exploratory Data Analysis

- Class distribution visualisation
- Vote pattern heatmap by party — reveals which votes most visibly separate Democrats and Republicans
- Abstention rate per feature

In [ ]:
# ── EDA: Class distribution + abstention rates ─────────────────────────────

dist = df['party'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class bar chart
axes[0].bar(
    [l.capitalize() for l in dist.index],
    dist.values,
    color=['#222222', '#888888'],
    edgecolor='white'
)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(dist.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

# Abstention rate per feature
VOTE_COLS = COLUMNS[1:]
abstain_rates = [(df[c] == '?').mean() * 100 for c in VOTE_COLS]
short_names = [c.replace('_', ' ').title()[:22] for c in VOTE_COLS]

axes[1].barh(short_names, abstain_rates, color='#555555', edgecolor='white')
axes[1].set_xlabel('Abstention Rate (%)')
axes[1].set_title('Abstention Rate per Vote', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ── EDA: Vote pattern heatmap by party ─────────────────────────────────────
# Encode: yes=1, no=0, ?=0.5 for visualisation only

def encode_for_viz(val):
    return 1.0 if val == 'y' else (0.5 if val == '?' else 0.0)

df_viz = df.copy()
for col in VOTE_COLS:
    df_viz[col] = df_viz[col].apply(encode_for_viz)

dem_means = df_viz[df_viz['party'] == 'democrat'][VOTE_COLS].mean()
rep_means = df_viz[df_viz['party'] == 'republican'][VOTE_COLS].mean()

heatmap_data = np.array([dem_means.values, rep_means.values])
short_names  = [c.replace('_', ' ').title()[:24] for c in VOTE_COLS]

fig, ax = plt.subplots(figsize=(16, 3))
im = ax.imshow(heatmap_data, cmap='Greys', aspect='auto', vmin=0, vmax=1)

ax.set_yticks([0, 1])
ax.set_yticklabels(['Democrat', 'Republican'], fontsize=11)
ax.set_xticks(range(len(VOTE_COLS)))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax.set_title('Mean Vote by Party  (1 = Yes · 0.5 = Abstain · 0 = No)', fontsize=12, fontweight='bold')

plt.colorbar(im, ax=ax, fraction=0.015, pad=0.04)
plt.tight_layout()
plt.show()

print('\nTop 5 most polarising votes (largest party difference):')
diff = abs(dem_means - rep_means).sort_values(ascending=False)
for feat, val in diff.head(5).items():
    print(f'  {feat}: {val:.3f}')

## 4. Preprocessing & Train-Test Split

**Encoding strategy:**
- `y` (yes) → 1
- `n` (no) → 0  
- `?` (abstain) → 2  (kept as a distinct third category, not imputed — abstention is itself informative)
- `party`: democrat → 0, republican → 1

**Feature representation:**
- Decision Tree: encoded integers (0, 1, 2) — no normalisation needed
- KNN: Hamming distance on encoded integers — no normalisation needed (all features on same scale)
- Naive Bayes: encoded integers used as categorical labels with frequency tables

**Split:** 80/20 stratified-order split with `random.seed(42)` — identical indices across all three models.

In [ ]:
# ── Encode dataset ──────────────────────────────────────────────────────────

VOTE_MAP  = {'y': 1, 'n': 0, '?': 2}
PARTY_MAP = {'democrat': 0, 'republican': 1}

df_enc = df.copy()
for col in VOTE_COLS:
    df_enc[col] = df_enc[col].map(VOTE_MAP)
df_enc['party'] = df_enc['party'].map(PARTY_MAP)

print('Encoding complete.')
print('Unique values per feature (should be 0, 1, 2):')
print(df_enc[VOTE_COLS].apply(lambda c: sorted(c.unique())).to_string())

In [ ]:
# ── Train-Test Split (80/20) ────────────────────────────────────────────────

indices = list(range(len(df_enc)))
random.seed(SEED)
random.shuffle(indices)

split     = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx  = indices[split:]

df_train = df_enc.iloc[train_idx].reset_index(drop=True)
df_test  = df_enc.iloc[test_idx].reset_index(drop=True)

X_train = df_train[VOTE_COLS].values.tolist()
y_train = df_train['party'].tolist()
X_test  = df_test[VOTE_COLS].values.tolist()
y_test  = df_test['party'].tolist()

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Train class dist : Democrat={y_train.count(0)}, Republican={y_train.count(1)}')
print(f'Test  class dist : Democrat={y_test.count(0)}, Republican={y_test.count(1)}')

## 5. Decision Tree — Implementation, Training & Feature Importance

The Decision Tree is built recursively using **Shannon entropy** and **Information Gain** as the split criterion.

**Stopping conditions:**
- Node is pure (all labels identical)
- Depth ≥ 8
- Node has fewer than 3 samples

**Feature Importance:** After training, each feature's total information gain contribution across all splits is accumulated — revealing which congressional votes are the strongest predictors of party affiliation.

| Symbol | Definition |
|--------|------------|
| H(S) | Shannon entropy of set S |
| IG(S, f, t) | Information gain from splitting S on feature f at threshold t |
| SL, SR | Left and right child partitions |

In [ ]:
# ── Decision Tree Implementation ────────────────────────────────────────────

class DecisionTree:
    def __init__(self, max_depth=8, min_samples=3):
        self.max_depth    = max_depth
        self.min_samples  = min_samples
        self.tree_        = None
        self.feature_importance_ = None

    # ── Entropy ──────────────────────────────────────────────────────────────
    def _entropy(self, y):
        total = len(y)
        if total == 0:
            return 0.0
        return -sum(
            (c / total) * math.log2(c / total)
            for c in Counter(y).values() if c > 0
        )

    # ── Information Gain ─────────────────────────────────────────────────────
    def _information_gain(self, y, y_left, y_right):
        n = len(y)
        return (
            self._entropy(y)
            - (len(y_left) / n) * self._entropy(y_left)
            - (len(y_right) / n) * self._entropy(y_right)
        )

    # ── Best Split ───────────────────────────────────────────────────────────
    def _best_split(self, X, y):
        best_gain, best_feat, best_thresh = -1, None, None
        n_feats = len(X[0])
        for feat in range(n_feats):
            thresholds = set(row[feat] for row in X)
            for thresh in thresholds:
                y_left  = [y[i] for i in range(len(X)) if X[i][feat] <= thresh]
                y_right = [y[i] for i in range(len(X)) if X[i][feat] >  thresh]
                if not y_left or not y_right:
                    continue
                gain = self._information_gain(y, y_left, y_right)
                if gain > best_gain:
                    best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh, best_gain

    # ── Recursive Build ──────────────────────────────────────────────────────
    def _build(self, X, y, depth, importance):
        if len(set(y)) == 1:
            return y[0]
        if depth >= self.max_depth or len(y) < self.min_samples:
            return Counter(y).most_common(1)[0][0]

        feat, thresh, gain = self._best_split(X, y)
        if feat is None:
            return Counter(y).most_common(1)[0][0]

        importance[feat] = importance.get(feat, 0.0) + gain

        left_mask  = [X[i][feat] <= thresh for i in range(len(X))]
        X_left  = [X[i] for i in range(len(X)) if     left_mask[i]]
        y_left  = [y[i] for i in range(len(y)) if     left_mask[i]]
        X_right = [X[i] for i in range(len(X)) if not left_mask[i]]
        y_right = [y[i] for i in range(len(y)) if not left_mask[i]]

        return {
            'feat'  : feat,
            'thresh': thresh,
            'left'  : self._build(X_left,  y_left,  depth + 1, importance),
            'right' : self._build(X_right, y_right, depth + 1, importance)
        }

    # ── Predict one ──────────────────────────────────────────────────────────
    def _predict_one(self, node, x):
        if not isinstance(node, dict):
            return node
        if x[node['feat']] <= node['thresh']:
            return self._predict_one(node['left'],  x)
        return self._predict_one(node['right'], x)

    # ── Public API ───────────────────────────────────────────────────────────
    def fit(self, X, y):
        importance     = {}
        self.tree_     = self._build(X, y, 0, importance)
        n_feats        = len(X[0])
        self.feature_importance_ = np.array(
            [importance.get(i, 0.0) for i in range(n_feats)]
        )
        total = self.feature_importance_.sum()
        if total > 0:
            self.feature_importance_ /= total
        return self

    def predict(self, X):
        return [self._predict_one(self.tree_, x) for x in X]

In [ ]:
# ── Train Decision Tree ─────────────────────────────────────────────────────

dt = DecisionTree(max_depth=8, min_samples=3)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)

print('Decision Tree training complete.')
print(f'Test predictions — Democrat: {dt_preds.count(0)}, Republican: {dt_preds.count(1)}')

In [ ]:
# ── Feature Importance ──────────────────────────────────────────────────────

imp    = dt.feature_importance_
order  = np.argsort(imp)[::-1]
short  = [c.replace('_', ' ').title()[:28] for c in VOTE_COLS]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(
    [short[i] for i in order],
    [imp[i]   for i in order],
    color='#333333', edgecolor='white'
)
ax.invert_yaxis()
ax.set_xlabel('Normalised Information Gain Contribution', fontsize=11)
ax.set_title('Decision Tree — Feature Importance\n(Which votes most strongly predict party affiliation?)',
             fontsize=12, fontweight='bold')

for bar, val in zip(bars, [imp[i] for i in order]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nTop 5 most predictive votes:')
for rank, i in enumerate(order[:5], 1):
    print(f'  {rank}. {VOTE_COLS[i]}: {imp[i]:.4f}')

## 6. K-Nearest Neighbours — Implementation, Training & k-Sensitivity Analysis

KNN stores the entire training set and classifies each test sample by majority vote among its k nearest neighbours.

**Distance metric — Hamming Distance:**  
Because all 16 features are categorical (encoded as 0, 1, 2), Euclidean distance is not appropriate — it would imply a false ordering relationship between vote values. Hamming distance counts the number of positions at which two vectors differ:

| Symbol | Definition |
|--------|------------|
| d_H(p, q) | Number of positions where p and q differ |
| k | Number of nearest neighbours |
| y_hat | Majority class among k neighbours |

**k-Sensitivity Analysis:** F1-Score is computed for k = 1 to 21 (odd values only) to identify the optimal k and show sensitivity to this hyperparameter.

In [ ]:
# ── KNN Implementation ──────────────────────────────────────────────────────

class KNN:
    def __init__(self, k=7):
        self.k          = k
        self.X_train_   = None
        self.y_train_   = None

    # ── Hamming distance ─────────────────────────────────────────────────────
    def _hamming(self, a, b):
        return sum(1 for x, y in zip(a, b) if x != y)

    # ── Public API ───────────────────────────────────────────────────────────
    def fit(self, X, y):
        self.X_train_ = X
        self.y_train_ = y
        return self

    def predict(self, X):
        preds = []
        for x in X:
            distances = [
                (self._hamming(x, self.X_train_[i]), self.y_train_[i])
                for i in range(len(self.X_train_))
            ]
            distances.sort(key=lambda t: t[0])
            k_labels = [label for _, label in distances[:self.k]]
            preds.append(Counter(k_labels).most_common(1)[0][0])
        return preds

In [ ]:
# ── Train KNN (k=7) ─────────────────────────────────────────────────────────

K = 7
knn = KNN(k=K)
knn.fit(X_train, y_train)
knn_preds = knn.predict(X_test)

print(f'KNN (k={K}) training complete.')
print(f'Test predictions — Democrat: {knn_preds.count(0)}, Republican: {knn_preds.count(1)}')

In [ ]:
# ── k-Sensitivity Analysis ──────────────────────────────────────────────────

def f1(y_true, y_pred, pos=1):
    TP = sum(1 for a, b in zip(y_true, y_pred) if a == pos and b == pos)
    FP = sum(1 for a, b in zip(y_true, y_pred) if a != pos and b == pos)
    FN = sum(1 for a, b in zip(y_true, y_pred) if a == pos and b != pos)
    p  = TP / (TP + FP) if (TP + FP) > 0 else 0
    r  = TP / (TP + FN) if (TP + FN) > 0 else 0
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

k_values = list(range(1, 22, 2))   # odd values 1–21
k_f1s    = []

print('Running k-sensitivity analysis...')
for k_val in k_values:
    preds = KNN(k=k_val).fit(X_train, y_train).predict(X_test)
    k_f1s.append(f1(y_test, preds))
    print(f'  k={k_val:>2}: F1 = {k_f1s[-1]:.4f}')

best_k   = k_values[np.argmax(k_f1s)]
best_f1  = max(k_f1s)
print(f'\nBest k: {best_k}  (F1 = {best_f1:.4f})')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_values, k_f1s, 'o-', color='#222222', linewidth=2, markersize=7)
ax.axvline(x=K,       color='#888888', linestyle='--', label=f'Chosen k={K}')
ax.axvline(x=best_k,  color='#444444', linestyle=':',  label=f'Best k={best_k}')
ax.set_xlabel('k (Number of Neighbours)', fontsize=11)
ax.set_ylabel('F1-Score', fontsize=11)
ax.set_title('KNN k-Sensitivity Analysis\n(Hamming Distance on Congressional Voting Data)',
             fontsize=12, fontweight='bold')
ax.set_xticks(k_values)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Naive Bayes — Implementation, Training & Smoothing Sensitivity

Naive Bayes estimates class-conditional probabilities from training frequency tables, with **Laplace smoothing** (α) to handle zero-frequency categories.

Because all features are already categorical (0 = No, 1 = Yes, 2 = Abstain), no binning is required — the raw encoded values serve directly as the vocabulary for each feature.

Classification uses log-probability summation to prevent numerical underflow:

| Symbol | Definition |
|--------|------------|
| P(C) | Prior probability of class C |
| P(xi = v \| C) | Laplace-smoothed likelihood of value v for feature i given class C |
| α | Laplace smoothing parameter |
| Vi | Vocabulary size for feature i |

**Smoothing Sensitivity Analysis:** Model accuracy is evaluated as α varies from 0.01 to 10, showing how over-smoothing degrades discriminative power on this dataset.

In [ ]:
# ── Naive Bayes Implementation ──────────────────────────────────────────────

class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha      = alpha
        self.priors_    = {}
        self.likelihoods_ = {}
        self.classes_   = None
        self.vocab_     = {}

    def fit(self, X, y):
        self.classes_ = list(set(y))
        n_feats       = len(X[0])
        n_total       = len(y)

        # Vocabulary per feature
        for feat in range(n_feats):
            self.vocab_[feat] = set(row[feat] for row in X)

        # Priors
        counts = Counter(y)
        self.priors_ = {c: counts[c] / n_total for c in self.classes_}

        # Likelihoods
        self.likelihoods_ = {c: {} for c in self.classes_}
        for c in self.classes_:
            subset = [X[i] for i in range(len(X)) if y[i] == c]
            nc     = len(subset)
            for feat in range(n_feats):
                vocab_size = len(self.vocab_[feat])
                feat_counts = Counter(row[feat] for row in subset)
                self.likelihoods_[c][feat] = {
                    v: (feat_counts.get(v, 0) + self.alpha) / (nc + self.alpha * vocab_size)
                    for v in self.vocab_[feat]
                }
        return self

    def _predict_one(self, x):
        log_probs = {}
        for c in self.classes_:
            log_p = math.log(self.priors_[c])
            for feat, val in enumerate(x):
                lk = self.likelihoods_[c][feat].get(
                    val,
                    self.alpha / (sum(self.likelihoods_[c][feat].values()) * len(self.vocab_[feat]) + self.alpha)
                )
                log_p += math.log(lk + 1e-12)
            log_probs[c] = log_p
        return max(log_probs, key=log_probs.get)

    def predict(self, X):
        return [self._predict_one(x) for x in X]

In [ ]:
# ── Train Naive Bayes (α=1.0) ───────────────────────────────────────────────

ALPHA = 1.0
nb    = NaiveBayes(alpha=ALPHA)
nb.fit(X_train, y_train)
nb_preds = nb.predict(X_test)

print(f'Naive Bayes (α={ALPHA}) training complete.')
print(f'Test predictions — Democrat: {nb_preds.count(0)}, Republican: {nb_preds.count(1)}')

In [ ]:
# ── Smoothing Sensitivity Analysis ─────────────────────────────────────────

def accuracy(y_true, y_pred):
    return sum(1 for a, b in zip(y_true, y_pred) if a == b) / len(y_true)

alpha_values = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0]
alpha_accs   = []

print('Running Laplace smoothing sensitivity analysis...')
for a in alpha_values:
    preds = NaiveBayes(alpha=a).fit(X_train, y_train).predict(X_test)
    acc   = accuracy(y_test, preds)
    alpha_accs.append(acc)
    print(f'  α={a:>5}: Accuracy = {acc:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alpha_values, alpha_accs, 's-', color='#222222', linewidth=2, markersize=7)
ax.axvline(x=ALPHA, color='#888888', linestyle='--', label=f'Chosen α={ALPHA}')
ax.set_xscale('log')
ax.set_xlabel('Laplace Smoothing Parameter (α) — log scale', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Naive Bayes — Smoothing Sensitivity Analysis\n(Effect of α on Classification Accuracy)',
             fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Evaluation & Results

### Metric Justification

The dataset has a moderate class imbalance (267 Democrats / 168 Republicans — a 61/39 split). Raw Accuracy is insufficient: a classifier always predicting Democrat would achieve ~61% without learning anything useful.

**F1-Score** is selected as the primary metric — as the harmonic mean of Precision and Recall, it penalises classifiers that sacrifice one for the other.

| Priority | Metric | Reason |
|----------|--------|--------|
| ★★★ | F1-Score | Balances Precision & Recall; primary metric |
| ★★ | Recall | Missing a Republican (FN) is costly for balanced representation |
| ★★ | Precision | False Republican label (FP) misrepresents voting patterns |
| ★ | Accuracy | Good overview; insufficient alone |

In [ ]:
# ── Evaluation Function ─────────────────────────────────────────────────────

def evaluate(name, y_true, y_pred, pos=1):
    TP = sum(1 for a, b in zip(y_true, y_pred) if a == pos and b == pos)
    TN = sum(1 for a, b in zip(y_true, y_pred) if a != pos and b != pos)
    FP = sum(1 for a, b in zip(y_true, y_pred) if a != pos and b == pos)
    FN = sum(1 for a, b in zip(y_true, y_pred) if a == pos and b != pos)

    acc  = (TP + TN) / (TP + TN + FP + FN)
    prec = TP / (TP + FP) if (TP + FP) > 0 else 0
    rec  = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    return {
        'Model': name, 'Accuracy': round(acc, 4), 'Precision': round(prec, 4),
        'Recall': round(rec, 4), 'F1-Score': round(f1, 4),
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN
    }

results = [
    evaluate('Decision Tree', y_test, dt_preds),
    evaluate(f'KNN (k={K})',   y_test, knn_preds),
    evaluate('Naive Bayes',   y_test, nb_preds),
]

results_df = pd.DataFrame(results)
print(results_df[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

In [ ]:
# ── Confusion Matrix Visualisation ─────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, r in zip(axes, results):
    cm = np.array([[r['TN'], r['FP']], [r['FN'], r['TP']]])
    im = ax.imshow(cm, cmap='Greys', aspect='auto')
    ax.set_title(r['Model'], fontsize=12, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred: Democrat', 'Pred: Republican'])
    ax.set_yticklabels(['Actual: Democrat', 'Actual: Republican'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black',
                    fontsize=14, fontweight='bold')

plt.suptitle('Confusion Matrices — 87-Sample Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Metric Comparison Bar Chart ─────────────────────────────────────────────

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors  = ['#222222', '#666666', '#aaaaaa']
x       = np.arange(len(metrics))
width   = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, (r, color) in enumerate(zip(results, colors)):
    vals = [r[m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=r['Model'],
                  color=color, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, color='black')

ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0.7, 1.08)
ax.legend()
plt.tight_layout()
plt.show()

print('\nBest model per metric:')
for m in metrics:
    best = max(results, key=lambda r: r[m])
    print(f'  {m:<12}: {best["Model"]} ({best[m]:.4f})')

## 9. K-Fold Cross-Validation

A single 80/20 split can produce results sensitive to that particular partition. **5-Fold Cross-Validation** provides a more robust estimate of generalisation performance by averaging F1-Score across 5 different train/test splits.

Mean and standard deviation are reported — a low standard deviation indicates stable, consistent performance across folds.

In [ ]:
# ── 5-Fold Cross-Validation ─────────────────────────────────────────────────

def k_fold_cv(model_fn, X, y, k=5):
    indices = list(range(len(X)))
    random.seed(SEED)
    random.shuffle(indices)

    fold_size = len(indices) // k
    fold_f1s  = []

    for fold in range(k):
        val_idx   = indices[fold * fold_size : (fold + 1) * fold_size]
        train_idx = indices[:fold * fold_size] + indices[(fold + 1) * fold_size:]

        X_tr  = [X[i] for i in train_idx]
        y_tr  = [y[i] for i in train_idx]
        X_val = [X[i] for i in val_idx]
        y_val = [y[i] for i in val_idx]

        model = model_fn()
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        fold_f1s.append(f1(y_val, preds))

    return fold_f1s

print('Running 5-Fold Cross-Validation across all three classifiers...\n')

X_all = df_enc[VOTE_COLS].values.tolist()
y_all = df_enc['party'].tolist()

dt_cv  = k_fold_cv(lambda: DecisionTree(max_depth=8, min_samples=3), X_all, y_all)
knn_cv = k_fold_cv(lambda: KNN(k=K),                                  X_all, y_all)
nb_cv  = k_fold_cv(lambda: NaiveBayes(alpha=ALPHA),                   X_all, y_all)

for name, folds in [('Decision Tree', dt_cv), (f'KNN (k={K})', knn_cv), ('Naive Bayes', nb_cv)]:
    print(f'{name:<18} — Mean F1: {np.mean(folds):.4f}  |  Std: {np.std(folds):.4f}  |  Folds: {[round(f, 4) for f in folds]}')

In [ ]:
# ── CV Results Visualisation ────────────────────────────────────────────────

cv_results  = [dt_cv, knn_cv, nb_cv]
cv_names    = ['Decision Tree', f'KNN (k={K})', 'Naive Bayes']
cv_means    = [np.mean(f) for f in cv_results]
cv_stds     = [np.std(f)  for f in cv_results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-fold F1 lines
for name, folds, color in zip(cv_names, cv_results, ['#222222', '#666666', '#aaaaaa']):
    axes[0].plot(range(1, 6), folds, 'o-', label=name, color=color, linewidth=2)
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('F1-Score')
axes[0].set_title('F1-Score per Fold', fontsize=12, fontweight='bold')
axes[0].set_xticks(range(1, 6))
axes[0].legend()
axes[0].grid(alpha=0.3)

# Mean ± Std bar chart
x_pos = np.arange(len(cv_names))
bars  = axes[1].bar(x_pos, cv_means, color=['#222222', '#666666', '#aaaaaa'],
                    edgecolor='white', yerr=cv_stds, capsize=6)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(cv_names)
axes[1].set_ylabel('Mean F1-Score')
axes[1].set_title('Mean F1 ± Std Dev (5-Fold CV)', fontsize=12, fontweight='bold')
axes[1].set_ylim(0.7, 1.05)
for bar, mean in zip(bars, cv_means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
                 f'{mean:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('5-Fold Cross-Validation Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Learning Curves

Learning curves show how each classifier's F1-Score changes as the training set size increases. They help diagnose:

- **Underfitting** — both training and validation scores are low
- **Overfitting** — large gap between training and validation scores
- **Good fit** — scores converge at a high value as training size grows

In [ ]:
# ── Learning Curves ─────────────────────────────────────────────────────────

def learning_curve(model_fn, X, y, sizes):
    random.seed(SEED)
    indices = list(range(len(X)))
    random.shuffle(indices)

    split     = int(0.8 * len(X))
    tr_idx    = indices[:split]
    val_idx   = indices[split:]

    X_val = [X[i] for i in val_idx]
    y_val = [y[i] for i in val_idx]

    train_f1s, val_f1s = [], []

    for size in sizes:
        tr_sub = tr_idx[:size]
        X_tr   = [X[i] for i in tr_sub]
        y_tr   = [y[i] for i in tr_sub]

        model = model_fn()
        model.fit(X_tr, y_tr)

        train_f1s.append(f1(y_tr,  model.predict(X_tr)))
        val_f1s.append(  f1(y_val, model.predict(X_val)))

    return train_f1s, val_f1s

sizes  = [30, 50, 80, 100, 130, 160, 200, 240, 280, 320, 348]
X_all  = df_enc[VOTE_COLS].values.tolist()
y_all  = df_enc['party'].tolist()

print('Generating learning curves...')
dt_tr,  dt_val  = learning_curve(lambda: DecisionTree(max_depth=8, min_samples=3), X_all, y_all, sizes)
knn_tr, knn_val = learning_curve(lambda: KNN(k=K),                                 X_all, y_all, sizes)
nb_tr,  nb_val  = learning_curve(lambda: NaiveBayes(alpha=ALPHA),                  X_all, y_all, sizes)
print('Done.')

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, name, tr, val in zip(
    axes,
    ['Decision Tree', f'KNN (k={K})', 'Naive Bayes'],
    [dt_tr,  knn_tr,  nb_tr],
    [dt_val, knn_val, nb_val]
):
    ax.plot(sizes, tr,  'o-',  color='#222222', label='Training F1',   linewidth=2)
    ax.plot(sizes, val, 's--', color='#888888', label='Validation F1', linewidth=2)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('F1-Score')
    ax.set_ylim(0.5, 1.1)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Learning Curves — All Classifiers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Final Comparison & Recommendations

In [ ]:
print('=' * 70)
print(f'{"FINAL MODEL COMPARISON":^70}')
print('=' * 70)
print(f'{"Model":<20} {"Accuracy":>10} {"Precision":>11} {"Recall":>8} {"F1-Score":>10}')
print('-' * 70)
for r in results:
    print(f'{r["Model"]:<20} {r["Accuracy"]:>10.4f} {r["Precision"]:>11.4f} {r["Recall"]:>8.4f} {r["F1-Score"]:>10.4f}')
print('=' * 70)

print(f"""
Primary Metric: F1-Score (justified by 61/39 class imbalance)

Key Findings:
  Feature Importance  — Physician Fee Freeze and El Salvador Aid are the
                        strongest party predictors by information gain.
  KNN Distance        — Hamming distance is correct for categorical voting
                        data; Euclidean would impose a false numeric ordering.
  Smoothing Effect    — Naive Bayes accuracy is robust across α = 0.1–2.0
                        but degrades at extremes (over- and under-smoothing).

Recommendations:
  Highest accuracy    → Decision Tree  (interpretable if-else rules on votes)
  Most stable (CV)    → Naive Bayes    (low std dev across folds)
  Lowest FN           → Decision Tree  (fewest Republicans missed)
""")